# Italian LLM Evaluation - Colab Quickstart

This notebook is the fast, beginner-facing companion to the full model-evaluation notebook. It explains the same configuration, execution, result, and interpretation concepts while using deliberately tiny defaults. The evaluation logic remains in the package; the notebook is an executable frontend.

Use this for:
- a fast smoke test on a small public causal LM
- a quick run on a Hugging Face model ID
- validating that the environment works before larger evaluations

By default LightEval is disabled, while BLiMP-IT uses 4 pairs, perplexity uses 3 streamed validation documents capped at 256 tokens, and generation uses 3 prompts under every configured decoding profile. These are integration checks, not publication measurements.


## What this notebook does

1. Clones your fork or the upstream repo
2. Installs the pinned runtime stack
3. Lets you choose a model source
4. Writes a temporary quick config
5. Runs tests
6. Runs the quick evaluation entry point
7. Explains and displays each evaluation family separately
8. Builds a cautious overall evidence card and shows how to continue to broader runs

`None` means unbounded for that particular enabled field. It does not enable a disabled component. Start here, then use `colab_model_eval_template.ipynb` for the guarded `smoke`, `broad`, `full`, and `custom` profile workflow.


In [ ]:
# Edit these values before running if needed.

REPO_URL = "https://github.com/GiorgosPeikos/it_eval_autoregressive_llms.git"  # Replace with your fork when needed.
REPO_DIR = "it_eval_autoregressive_llms"

# Set to a public HF repo id for a quick smoke run.
# For local checkpoints on Drive, use a mounted path like /content/drive/MyDrive/models/your-checkpoint
MODEL_SOURCE = "Gpeik/Sophira-360M-base"
MODEL_REVISION = None  # Automatically resolve the model repository's current commit SHA.
TOKENIZER_SOURCE = "Gpeik/Sophira-360M-base"
TOKENIZER_REVISION = None  # Resolve independently when TOKENIZER_SOURCE is another repository.
TRUST_REMOTE_CODE = False  # Enable only for a reviewed repository requiring custom model code.
TOKENIZER_USE_FAST = True
MAX_MODEL_LENGTH = None  # Optional context-length override.
ARTIFACT_SHA256 = None  # Recommended identity digest for a local checkpoint.

# Keep this small for fast validation.
ENABLE_LIGHTEVAL = False
LIGHTEVAL_SUITE = "quick"  # quick, full, verified_windows, or all.
MAX_LIGHTEVAL_SAMPLES = 2
MAX_BLIMP_SAMPLES = 4
ENABLE_BLIMP_IT = True
ENABLE_PERPLEXITY = True
ENABLE_GENERATION = True
OVERWRITE_RESULTS = False
SAVE_DETAILS = True  # Disable for a smaller LightEval output bundle.

# Perplexity corpus and evaluation budget.
PPL_DATASET_REPO = "gsarti/clean_mc4_it"
PPL_DATASET_SUBSET = "tiny"  # clean_mc4_it: tiny=1/8, small=1/4, medium=1/2, large=3/4, full=all.
PPL_DATASET_SPLIT = "validation"
PPL_DATASET_STREAMING = True  # Avoid downloading/materializing unused splits.
MAX_PPL_DOCUMENTS = 3  # Set to None to score the complete selected subset.
MAX_PPL_TOKENS_PER_DOCUMENT = 256
PPL_SEQUENCE_LENGTH = 128
PPL_STRIDE = 64
PPL_PRESERVE_DOCUMENT_BOUNDARIES = True
PPL_ADD_BOS_TOKEN = True
PPL_ADD_EOS_TOKEN = False
PPL_PER_DOCUMENT_STATS = True
MAX_GENERATION_PROMPTS = 3
GENERATION_PROFILES = [
    {"name": "greedy", "do_sample": False, "max_new_tokens": 128},
    {"name": "temp_0_7_top_p_0_9", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "max_new_tokens": 128},
    {"name": "temp_0_8_top_p_0_95", "do_sample": True, "temperature": 0.8, "top_p": 0.95, "max_new_tokens": 128},
]

# Leave as None to auto-select cuda when Colab exposes a GPU, otherwise cpu.
MODEL_DEVICE = None
MODEL_DTYPE = "auto"
MODEL_BATCH_SIZE = 1
PARALLELISM = "auto"  # Uses replicated inference when the runtime exposes multiple GPUs.
NUM_PROCESSES = "auto"  # Or set an integer no larger than the visible GPU count.

# Optional HF token for gated/private models.
HF_TOKEN = ""
MOUNT_GOOGLE_DRIVE = False
OUTPUT_ROOT = "evaluation_results"
RANDOM_SEED = 13


In [ ]:
import os
import shutil
from pathlib import Path

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

%cd /content

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

repo_path = Path("/content") / REPO_DIR
if repo_path.exists():
    shutil.rmtree(repo_path)

!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/{REPO_DIR}

In [ ]:
!python --version

## Install the pinned stack

The repository targets Python 3.10 to 3.13 for the LightEval plus legacy-dataset path. Colab normally provides a compatible Python runtime.


In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel
!python -m pip install "lighteval[multilingual]==0.13.0" --no-deps
!python -m pip install -r constraints/lighteval-python310-313.txt
!python -m pip install -e .[dev] --no-deps

In [ ]:
import torch
from it_eval_framework.utils.lighteval_runtime import lighteval_environment_report

if ENABLE_LIGHTEVAL:
    lighteval_report = lighteval_environment_report()
    print(f"LightEval preflight: {lighteval_report}")
    if lighteval_report["errors"]:
        raise RuntimeError("LightEval preflight failed; rerun the installation cell.")

detected_device = "cuda" if torch.cuda.is_available() else "cpu"
selected_device = MODEL_DEVICE or detected_device

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"cuda device count: {torch.cuda.device_count()}")
    print(f"cuda device name: {torch.cuda.get_device_name(0)}")
    print(f"cuda capability: {torch.cuda.get_device_capability(0)}")
else:
    print("No CUDA GPU is available in this runtime. In Colab, switch Runtime > Change runtime type > GPU.")


In [ ]:
from pathlib import Path
import yaml

config_path = Path("configs/colab_quickstart.yaml")
config_payload = {
    "run_name": "colab_quickstart",
    "model": {
        "source": MODEL_SOURCE,
        "revision": MODEL_REVISION,
        "tokenizer_source": TOKENIZER_SOURCE or MODEL_SOURCE,
        "tokenizer_revision": TOKENIZER_REVISION,
        "dtype": MODEL_DTYPE,
        "device": selected_device,
        "batch_size": MODEL_BATCH_SIZE,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "tokenizer_use_fast": TOKENIZER_USE_FAST,
        "max_model_length": MAX_MODEL_LENGTH,
        "artifact_sha256": ARTIFACT_SHA256,
    },
    "output": {
        "root_dir": OUTPUT_ROOT,
        "overwrite": OVERWRITE_RESULTS,
        "save_details": SAVE_DETAILS,
    },
    "runtime": {
        "seed": RANDOM_SEED,
        "python_executable": "python",
        "lighteval_command": "lighteval",
        "parallelism": PARALLELISM,
        "num_processes": NUM_PROCESSES,
    },
    "lighteval": {
        "enabled": ENABLE_LIGHTEVAL,
        "suite": LIGHTEVAL_SUITE if ENABLE_LIGHTEVAL else None,
        "max_samples": MAX_LIGHTEVAL_SAMPLES,
        "dataset_loading_processes": 1,
        "num_fewshot_seeds": 1,
        "extra_args": [],
    },
    "blimp_it": {
        "enabled": ENABLE_BLIMP_IT,
        "dataset_revision": "4159ecb68388283488cb1d235a7e1946489bc62d",
        "max_samples": MAX_BLIMP_SAMPLES,
    },
    "perplexity": {
        "enabled": ENABLE_PERPLEXITY,
        "dataset_repo": PPL_DATASET_REPO,
        "dataset_subset": PPL_DATASET_SUBSET,
        "dataset_revision": "167d5696e91ac89f17936f9d0059031cbc4c9e99",
        "dataset_trust_remote_code": True,
        "dataset_streaming": PPL_DATASET_STREAMING,
        "split": PPL_DATASET_SPLIT,
        "text_field": "text",
        "sequence_length": PPL_SEQUENCE_LENGTH,
        "stride": PPL_STRIDE,
        "preserve_document_boundaries": PPL_PRESERVE_DOCUMENT_BOUNDARIES,
        "add_bos_token": PPL_ADD_BOS_TOKEN,
        "add_eos_token": PPL_ADD_EOS_TOKEN,
        "per_document_stats": PPL_PER_DOCUMENT_STATS,
        "max_documents": MAX_PPL_DOCUMENTS,
        "max_tokens_per_document": MAX_PPL_TOKENS_PER_DOCUMENT,
    },
    "generation": {
        "enabled": ENABLE_GENERATION,
        "prompts_path": "configs/generation_prompts.yaml",
        "max_prompts": MAX_GENERATION_PROMPTS,
        "seed": RANDOM_SEED,
        "profiles": GENERATION_PROFILES,
    },
}

with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config_payload, handle, allow_unicode=True, sort_keys=False)

print(config_path)
print(config_path.read_text(encoding="utf-8"))

## Understand the generated configuration

The printed YAML is the same validated schema used by the CLI and Python API. Model and tokenizer revisions are resolved independently to immutable commits. The run log prints every enabled component, selected LightEval alias and resolved task, dataset split, and applicable limit.

| Block | Key controls | Scope |
|---|---|---|
| `model` | sources, revisions, dtype, device, batch size | Which checkpoint/tokenizer is loaded and how |
| `output` | overwrite, save details | Resume behavior and sample-level artifacts |
| `runtime` | seed, parallelism, processes | Reproducibility and GPU execution |
| `lighteval` | enabled, suite, max samples | Tasks plus a **per-task** inference cap |
| `blimp_it` | enabled, max samples | Global cap across selected linguistic subsets |
| `perplexity` | corpus, split, windows, documents/tokens | Exact held-out token stream and compute budget |
| `generation` | prompts, cap, profiles | Prompt/profile combinations to generate |

LightEval suites are `quick` (1 task), `full` (33-task curated set), `verified_windows` (39 retained variants), and `all` (all 39 currently evaluable Italian variants). `mkqa.long_answer` remains addressable explicitly but is excluded from executable sweeps because the pinned stack yields no evaluation documents. A `None` limit means all available items for that field.


## Validate the installed library

This test cell checks the repository before spending GPU time. It is useful after a fresh clone or dependency change and can be skipped on a repeated run when the same commit and environment already passed.


In [ ]:
!python -m pytest

## Run the bounded evaluation

`it-eval-launch` selects single- or multi-GPU execution from the configuration, resolves revisions, prints the evaluation plan, resumes compatible completed stages when possible, runs every enabled component, and writes normalized `summary.csv` plus the detailed artifacts. The quick defaults are deliberately too small for model-quality claims.


In [ ]:
!it-eval-launch --config configs/colab_quickstart.yaml

## Evaluation results

The next cell shows stage completion, metrics for every enabled evaluation, the rendered report, and a bounded preview of generated continuations.


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import pandas as pd
from it_eval_framework.reporting.metric_semantics import annotate_metric_rows

run_configs = list(Path(OUTPUT_ROOT).rglob("run_config.yaml"))
if not run_configs:
    raise FileNotFoundError("No evaluation run was found. Run the evaluation cell first.")
run_dir = max(run_configs, key=lambda path: path.stat().st_mtime).parent
display(Markdown(f"**Complete result directory:** `{run_dir}`"))
resolved_config_path = run_dir / "resolved_config.yaml"
if resolved_config_path.exists():
    display(Markdown("### Resolved configuration (what was actually evaluated)"))
    display(Markdown(f"```yaml\n{resolved_config_path.read_text(encoding='utf-8')}\n```"))

state_path = run_dir / "run_state.json"
if state_path.exists():
    steps = json.loads(state_path.read_text(encoding="utf-8")).get("steps", {})
    status_rows = [{"stage": stage, **details} for stage, details in steps.items()]
    display(Markdown("### Stage status"))
    display(pd.DataFrame(status_rows).fillna("—"))

summary_path = run_dir / "summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"The run has no summary yet: {summary_path}")
summary = pd.read_csv(summary_path)
component_labels = {"blimp_it": "BLiMP-IT", "perplexity": "Perplexity", "generation": "Generation", "lighteval": "LightEval"}
def show_component(component):
    metrics = summary[summary["component"] == component]
    if metrics.empty:
        display(Markdown(f"*{component_labels.get(component, component)} was disabled or did not complete.*"))
        return
    display(Markdown(f"### {component_labels.get(component, component)} metrics"))
    visible = annotate_metric_rows(metrics).drop(columns=["component"]).dropna(axis=1, how="all").reset_index(drop=True)
    display(visible)
    if visible["metric"].astype(str).str.contains("acc|exact_match|f1", case=False, regex=True).any():
        display(Markdown("A value of **1.0** on a 0–1 accuracy-like metric means every evaluated item received a score of 1; **0.75 means 75%**. Always read `sample_count`: 1.0 on 2 examples is not evidence of perfect dataset-wide performance."))

report_path = run_dir / "report.md"
generations_path = run_dir / "generations.jsonl"
def show_generation_preview():
    if not generations_path.exists():
        display(Markdown("*Generation was disabled or did not complete.*"))
        return
    generation_rows = [json.loads(line) for line in generations_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    preview = pd.DataFrame([{
        "prompt_id": row.get("prompt_identifier"),
        "profile": row.get("decoding_profile", {}).get("name"),
        "prompt": row.get("prompt_text"),
        "generated_text": row.get("generated_text", "")[:500],
        "words": row.get("output_length_words"),
        "distinct_2": row.get("distinct_2"),
        "repeated_3gram_rate": row.get("repeated_3gram_rate"),
        "unfinished": row.get("unfinished_output"),
        "artifacts": row.get("artifact_flags"),
    } for row in generation_rows[:6]])
    display(Markdown(f"### Generation preview ({len(preview)} of {len(generation_rows)})"))
    with pd.option_context("display.max_colwidth", 500):
        display(preview)

## LightEval benchmark results

LightEval is disabled by default in this quickstart because it adds an optional benchmark runtime and dataset preparation. Set `ENABLE_LIGHTEVAL = True` to exercise it. The `quick` suite selects one Italian task; `all` selects all 39 currently evaluable Italian task variants. `MAX_LIGHTEVAL_SAMPLES` applies separately to every selected task.

For accuracy-like metrics, every evaluated item contributes 1 when correct and 0 otherwise, and the value is their mean. Thus `1.0` means all **evaluated** examples were correct—not universal perfection. Interpret `cf`, `mcf`, and `hybrid` as different formulations, and always read the sample count, few-shot count, and standard error.


In [ ]:
show_component("lighteval")

## BLiMP-IT grammatical preference results

BLiMP-IT compares each grammatical Italian sentence with a minimally different ungrammatical sentence. The framework sums next-token log-probabilities and counts the pair as correct when the grammatical member receives at least as much probability. Overall accuracy is `correct pairs / evaluated pairs`; phenomenon rows use the same calculation within one linguistic category.

Accuracy `1.0` means all evaluated pairs were ordered correctly. With the default cap of four pairs, it is only proof that this pipeline stage works. It does not establish general grammatical competence or free-generation quality.


In [ ]:
show_component("blimp_it")

## Held-out perplexity results

Mean loss is total negative log-likelihood divided by scored target tokens; token perplexity is `exp(mean loss)`. Lower is better, and `1.0` is the theoretical lower limit—not 100% accuracy.

Only compare perplexity when tokenizer, corpus revision/subset/split, preprocessing, sequence length, stride, boundary policy, and budgets match. Streaming prevents the quickstart from materializing unused remote splits. For research claims, verify that the corpus was not included in model training.


In [ ]:
show_component("perplexity")

## Controlled-generation results

Every selected prompt is run under every decoding profile. `num_generations` is coverage, not quality. `distinct_2` describes local diversity, repeated-trigram rate flags repetition, `unfinished_output` is a punctuation heuristic, and artifact flags detect possible corruption or degeneration.

Inspect representative outputs from every prompt category/profile. Human review is required for Italian fluency, coherence, factuality, relevance, safety, toxicity, and bias.


In [ ]:
show_component("generation")
show_generation_preview()

## Overall model assessment

There is no scientifically valid universal total obtained by averaging these components: benchmark correctness, grammatical preference, predictive fit, and generation diagnostics use incompatible scales. The next cell builds a multidimensional evidence card and labels the quick run as diagnostic.

A model-selection claim requires a named baseline evaluated with the same resolved configuration, uncertainty-aware comparisons, contamination checks, and human review of generations.


In [ ]:
assessment = []
observed_components = set(summary["component"].astype(str))
assessment.append(f"**Coverage:** {len(observed_components)}/4 evaluation families produced metrics ({', '.join(sorted(observed_components))}).")
light = summary[summary["component"] == "lighteval"]
light_accuracy = light[light["metric"].astype(str).str.contains("acc|exact_match|f1", case=False, regex=True)]
if not light_accuracy.empty:
    assessment.append(f"**LightEval:** {len(light_accuracy)} accuracy-like rows; observed range {light_accuracy['value'].min():.4f}–{light_accuracy['value'].max():.4f}. This mixes tasks/formulations and is not a combined score.")
else:
    assessment.append("**LightEval:** not evaluated in the default quick run.")
blimp = summary[(summary["component"] == "blimp_it") & (summary["task_id"] == "all") & (summary["metric"] == "accuracy")]
if not blimp.empty:
    row = blimp.iloc[0]
    assessment.append(f"**BLiMP-IT:** accuracy {row['value']:.4f} across {int(row['sample_count'])} evaluated pairs.")
ppl = summary[(summary["component"] == "perplexity") & (summary["metric"] == "token_perplexity")]
if not ppl.empty:
    row = ppl.iloc[0]
    assessment.append(f"**Perplexity:** {row['value']:.4f} across {int(row['sample_count'])} target tokens; lower is better only under identical settings.")
if generations_path.exists():
    assessed_generations = [json.loads(line) for line in generations_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if assessed_generations:
        repetition = pd.Series([row.get("repeated_3gram_rate") for row in assessed_generations], dtype="float64").mean()
        unfinished = sum(bool(row.get("unfinished_output")) for row in assessed_generations)
        assessment.append(f"**Generation:** {len(assessed_generations)} outputs; mean repeated-3-gram rate {repetition:.4f}; {unfinished} flagged unfinished. These are review signals, not a grade.")
assessment.append("**Assessment boundary:** this deliberately tiny quick run validates integration. It cannot support a final model-quality claim.")
display(Markdown("\n\n".join(assessment)))

## Download the reproducibility bundle

Keep the complete run directory—not only `summary.csv`. It includes resolved revisions, configuration, environment, state, raw/component results, sample details when enabled, and the rendered report.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("italian_model_quick_evaluation", "zip", root_dir=run_dir)
files.download(archive)

## Next steps after the quick run

- Replace `MODEL_SOURCE` and `TOKENIZER_SOURCE` to evaluate another Hub model or mounted local checkpoint.
- Enable LightEval first with `suite=quick`, then progress to bounded `all` before removing sample caps.
- Open `colab_model_eval_template.ipynb` for the guarded `smoke`, `broad`, `full`, and editable `custom` profiles.
- Replace the example perplexity corpus with genuinely held-out Italian text for publication-grade claims.
- Compare checkpoints only with identical evaluation settings using `it-eval-compare`.
- Use `it-eval-check-lighteval`, component-specific `it-eval-run-*` commands, and the task probe runner for diagnosis.
- Read `docs/RESULTS_README.md`, `docs/TASK_REFERENCE.md`, `docs/LIGHTEVAL_README.md`, and `docs/MULTI_GPU_README.md` for formulas, datasets, troubleshooting, and distributed execution.
